# Unit 4 solutions: co-occurrence, item-item cosine, and k-NN collaborative filtering

These reference solutions rebuild the first *collaborative* path from the lesson -- a binary reader x item incidence of the train positives, an item-item cosine co-occurrence similarity, then k-nearest-neighbour retrieval over it -- reveal the `bookrec` implementation, and score it honestly on the Unit 1 scoreboard against the random floor, popularity, and the Unit-3 lexical path. Collaborative filtering reads only *who read what*; it never opens the catalog text. A reader's `seen` set is always their positive `train` history, and `val` is the sealed holdout we grade against. The only random path is the `RandomRetrievalPath(..., seed=0)` floor, so the notebook runs top-to-bottom deterministically, and every exercise ends with an `assert` on a computed result so a wrong number fails loudly instead of printing quietly.

In [ ]:
import json

import bookrec
import numpy as np
import pandas as pd

data_dir = bookrec.generated_dir()
catalog = bookrec.load_catalog(data_dir / "catalog.csv.gz")
catalog_ids = sorted(catalog)
interactions_path = data_dir / "interactions.csv.gz"
interactions = pd.read_csv(interactions_path)
rows = interactions.to_dict("records")

cold = json.loads((data_dir / "cold_partitions.json").read_text())
cold_readers = set(cold["cold_readers"])

# The train positives are both the co-occurrence signal and each reader's `seen` set.
train = interactions[interactions["split"] == "train"]
train_positive = train[train["label"] == 1]
seen_by = (
    train_positive.groupby("reader_id")["item_id"]
    .apply(lambda items: {int(i) for i in items})
    .to_dict()
)

# Map each catalog book to its column in the reader x item matrices below.
col_of = {item_id: col for col, item_id in enumerate(catalog_ids)}

len(catalog_ids), len(interactions), len(train)

## Exercise 1 - Implicit feedback by the numbers: positives, exposure-sampled negatives, and what co-occurrence reads

The train log mixes two kinds of implicit-feedback row, and co-occurrence reads only one of them. A `label == 1` row is a clear **positive** (the reader engaged the book); a `label == 0` row is an **exposure-sampled negative** (the reader was shown the book and did not engage it), a stand-in for the unobserved "did not like" -- there are no explicit negative *ratings* here. Counting the train split three ways shows positives are only about a third of it. The item-item co-occurrence similarity is built from **positives only** (`split == "train"` and `label == 1`) and ranks the full catalog, so exactly `n_positive` rows feed it and the `n_negative` sampled negatives are ignored; the two groups partition the train log, so `n_positive + n_negative == n_train`.

In [ ]:
n_train = len(train)                                   # number of rows in the train split
n_positive = int((train["label"] == 1).sum())          # train rows with label == 1 (engaged)
n_negative = int((train["label"] == 0).sum())          # train rows with label == 0 (exposed, not engaged)
positive_share = n_positive / n_train

# These describe the SAME train log; the co-occurrence similarity reads only the positives.
rows_entering_cooccurrence = n_positive                # positives feed the item-item similarity
rows_ignored_by_cooccurrence = n_negative              # the sampled negatives are ignored in Unit 4

positives_only_explanation = (
    "Only label==1 rows carry a co-LIKING signal: two books are neighbours because the same readers "
    "liked both, and a label==0 (exposed-but-not-engaged) row is a sampled guess at a dislike, not a "
    "shared like, so it says nothing about which books belong together. Unit 4 therefore does NO "
    "negative sampling at all -- it counts co-occurrence among positives and ranks the full catalog. "
    "The sampled negatives are genuinely useful, but for a different job (telling positives from "
    "negatives), which is exactly what Unit 5 (matrix factorization) and Unit 8 (the two-tower model) "
    "use them for as training signal."
)

print(f"train rows: {n_train}")
print(f"  label==1 positives        (engaged)            : {n_positive}")
print(f"  label==0 sampled negatives (exposed, not engaged): {n_negative}")
print(f"positives are {positive_share:.1%} of the train log")
print(f"rows feeding co-occurrence: {rows_entering_cooccurrence}   ignored: {rows_ignored_by_cooccurrence}")

assert n_positive + n_negative == n_train
assert rows_entering_cooccurrence == n_positive and rows_ignored_by_cooccurrence == n_negative
assert 0.30 < positive_share < 0.40, "positives are about a third of the train log"


## Exercise 2 - Build the binary reader x item incidence of the train positives

Collaborative filtering starts from a binary **reader x item incidence** matrix $\mathbf{R}$, where $\mathbf{R}_{r,i} = 1$ exactly when reader $r$ has a train positive for item $i$. We collect the distinct `(reader_id, item_id)` positive pairs, index the readers that appear in sorted order (readers are the rows), and take the **whole catalog** as the item universe (columns, via `col_of`). The number of non-zero entries equals the number of distinct positive pairs, and the column sum for book $307$ reads back in plain English as the count of readers who liked it -- two of them on this slice.

In [ ]:
pairs = set(zip(train_positive["reader_id"].astype(int), train_positive["item_id"].astype(int)))
reader_ids = sorted({reader for reader, _ in pairs})
reader_of = {reader: row for row, reader in enumerate(reader_ids)}

incidence = np.zeros((len(reader_ids), len(catalog_ids)))
for reader, item_id in pairs:
    incidence[reader_of[reader], col_of[item_id]] = 1.0

incidence_shape = incidence.shape                      # (readers, items)
n_pairs = int(incidence.sum())
readers_of_book_307 = int(incidence[:, col_of[307]].sum())   # how many readers liked book 307
sum_matches_pairs = int(incidence.sum()) == len(pairs)

print(f"incidence matrix {incidence_shape}  (readers x items)")
print(f"non-zero entries (reader-liked-book pairs): {n_pairs}  == #pairs: {sum_matches_pairs}")
print(f"readers who liked book 307: {readers_of_book_307}")

assert sum_matches_pairs and n_pairs == len(pairs)
assert incidence_shape == (len(reader_ids), len(catalog_ids))
assert set(np.unique(incidence)) <= {0.0, 1.0}, "incidence is binary"
assert readers_of_book_307 == 2


## Exercise 3 - The item-item cosine co-occurrence similarity, by hand, then reveal `bookrec.item_item_cosine`

Two books co-occur for every reader who liked both -- the dot product of their incidence columns, the plain count of shared readers. Raw co-occurrence favours popular books, so we normalize by each column's length, exactly the cosine between two *item* columns:

$$\mathrm{sim}(i, j) = \frac{\mathbf{R}_{:,i} \cdot \mathbf{R}_{:,j}}{\lVert \mathbf{R}_{:,i} \rVert \, \lVert \mathbf{R}_{:,j} \rVert}.$$

The numerator is the item-item dot product $\mathbf{R}^{\top}\mathbf{R}$ (every shared-reader count at once); the denominator is the outer product of the column norms clamped to a tiny floor, so a book nobody read (norm $0$) gives an all-zero row instead of dividing by zero. We zero the diagonal so a book is never its own neighbour. Books $307$ and $1066$ share exactly one reader and their cosine is $0.5$. Revealing `bookrec.item_item_cosine(incidence)` shows the package ships exactly this computation, agreeing to floating-point precision.

In [ ]:
norms = np.sqrt((incidence ** 2).sum(axis=0))          # ||column_i|| = sqrt(#readers of i)
sim_by_hand = (incidence.T @ incidence) / np.maximum(np.outer(norms, norms), 1e-9)
np.fill_diagonal(sim_by_hand, 0.0)                     # a book is not its own neighbour

i, j = col_of[307], col_of[1066]
shared_readers = int((incidence[:, i] * incidence[:, j]).sum())   # co-readers of 307 and 1066
sim_307_1066 = sim_by_hand[i, j]
diagonal_is_zero = bool(np.allclose(np.diag(sim_by_hand), 0.0))
is_symmetric = bool(np.allclose(sim_by_hand, sim_by_hand.T))

sim_lib = bookrec.item_item_cosine(incidence)
matches_library = bool(np.allclose(sim_lib, sim_by_hand))

print(f"books 307 and 1066 share {shared_readers} reader(s); cosine sim = {sim_307_1066:.3f}")
print(f"diagonal all zero: {diagonal_is_zero}   symmetric: {is_symmetric}")
print(f"by-hand cosine == bookrec.item_item_cosine: {matches_library}")

assert matches_library, "by-hand cosine must equal bookrec.item_item_cosine to floating point"
assert diagonal_is_zero and is_symmetric
assert shared_readers == 1 and np.isclose(sim_307_1066, 0.5)


## Exercise 4 - k-nearest-neighbour retrieval: score candidates by summed similarity to a reader's history

To recommend *for a reader*, their history is the query: a candidate book scores by the **summed similarity** to every book they have already liked,

$$\mathrm{score}(c) = \sum_{s \,\in\, \mathrm{seen}} \mathrm{sim}(c, s).$$

We gather the columns of reader $3$'s seen books, sum the similarity rows across them to score every catalog book, wrap each book as a `bookrec.Candidate`, and call `bookrec.rank(..., n=5, exclude=seen)`. The `exclude=seen` drops the books the reader has already read; the `n=5` here is how many recommendations to return, not the `n_neighbors` cap of the challenge exercises. None of the returned ids is in `seen`.

In [ ]:
reader_id = 3
seen = seen_by[reader_id]                              # the reader's train-positive books
seen_cols = [col_of[item_id] for item_id in sorted(seen)]
scores = sim_by_hand[:, seen_cols].sum(axis=1)         # summed similarity to the history

candidates = [
    bookrec.Candidate(item_id, float(scores[col]), "cooc-by-hand")
    for col, item_id in enumerate(catalog_ids)
]
top5 = bookrec.rank(candidates, n=5, exclude=seen)
top5_ids = [c.item_id for c in top5]
no_seen_recommended = all(item_id not in seen for item_id in top5_ids)
exclude_explanation = (
    "A re-read is not a recommendation -- the reader has already engaged those books -- so surfacing a "
    "book they have read wastes a slot and cannot earn a hit on the held-out val positives, which are "
    "books they had NOT yet read in train."
)

print(f"reader {reader_id} has read {sorted(seen)}")
print(f"top-5 recommendations: {top5_ids}   none in seen: {no_seen_recommended}")

assert no_seen_recommended
assert len(top5_ids) == 5 and len(set(top5_ids)) == 5
assert top5_ids == [976, 1584, 113, 1767, 1586]


## Exercise 5 - Reveal `ItemItemRetrievalPath`: reproduce the by-hand ranking, and the honest empty-history answer

`bookrec.ItemItemRetrievalPath()` packages the by-hand pieces as a standard retrieval path. Its `fit(interactions, catalog=None)` duck-types the row-mapping log exactly like `PopularityRetrievalPath` (no pandas inside the package) and honours `catalog=` as the item universe; `retrieve(query, context, k)` identifies the reader through `context["seen"]`, scores each candidate by the summed similarity to the seen books, excludes the seen books, and returns the top `k`. Fit on `rows` with `catalog=catalog_ids`, it reproduces the by-hand `top5_ids` from Exercise 4 (same ids, same order). A reader with an **empty** `seen` set gets `[]` -- the honest answer from a path that depends entirely on history.

In [ ]:
cf = bookrec.ItemItemRetrievalPath().fit(rows, catalog=catalog_ids)
shipped = cf.retrieve(query=reader_id, context={"seen": seen}, k=5)
shipped_ids = [c.item_id for c in shipped]
reproduces_by_hand = shipped_ids == top5_ids

empty_history = cf.retrieve(query=999_999, context={"seen": set()}, k=5)
empty_is_empty = empty_history == []
cold_start_explanation = (
    "The Unit 2 PopularityRetrievalPath steps in for a reader with no history: it needs no personal "
    "signal at all and recommends the globally most-engaged books, a sensible non-personalized default "
    "until the reader has a `seen` set for CF to work from."
)

print(f"shipped top-5: {shipped_ids}   reproduces by-hand: {reproduces_by_hand}")
print(f"empty history -> {empty_history}   is []: {empty_is_empty}")

assert reproduces_by_hand, "the shipped path must reproduce the by-hand ranking exactly"
assert empty_is_empty


## Exercise 6 - Register the paths and read the validation scoreboard honestly

CF lines up against the earlier units' paths under the Unit 1 rules unchanged: a seeded random floor, popularity (Unit 2), and lexical (Unit 3), all registered in one `PathRegistry` and scored on `val` with `k=10` and `cold_readers` excluded. CF scores about **0.252** hit@10 -- roughly **2.3x** popularity (~0.108), **1.6x** the lexical path (~0.158), and **21x** the random floor (~0.012).

In [ ]:
keywords = bookrec.load_keywords(data_dir / "keywords.csv.gz")
random_floor = bookrec.RandomRetrievalPath(catalog_ids, seed=0)
popularity = bookrec.PopularityRetrievalPath().fit(rows, catalog=catalog_ids)
lexical = bookrec.LexicalRetrievalPath(keywords).fit(rows, catalog=catalog_ids)

registry = bookrec.PathRegistry()
for path in (random_floor, popularity, lexical, cf):
    registry.register(path)


def score(path):
    return bookrec.run_validation_scoreboard(
        path, interactions_path, catalog_ids=catalog_ids, k=10, cold_readers=cold_readers
    )


cf_hit = score(cf).hit_rate_at_k
popularity_hit = score(popularity).hit_rate_at_k
lexical_hit = score(lexical).hit_rate_at_k
floor_hit = score(random_floor).hit_rate_at_k
cf_vs_popularity = cf_hit / popularity_hit             # expect ~2.3
cf_vs_lexical = cf_hit / lexical_hit                   # expect ~1.6
cf_vs_floor = cf_hit / floor_hit                       # expect ~21
scoreboard_reading = (
    "CF is NOT the first path to beat popularity -- Unit 3's lexical path already modestly beat it "
    "(~1.46x), and the lesson said so. CF is the first COLLABORATIVE path (its signal is other readers, "
    "not the item's own text) and the first to beat BOTH popularity and the content/lexical path, by the "
    "widest margin so far (~2.3x over popularity, ~1.6x over lexical). The ratios establish that reading "
    "who-read-what carries more signal here than the crowd's totals or the books' words; they do NOT make "
    "CF a strong recommender in absolute terms -- 0.252 hit@10 still misses about three readers in four -- "
    "and they say nothing about the cold ~41% of the catalog CF cannot reach at all."
)

print(f"{'path':>14} {'hit@10':>8} {'vs floor':>9}")
for label, hit in (("random", floor_hit), ("popularity", popularity_hit),
                   ("lexical", lexical_hit), ("item-item CF", cf_hit)):
    print(f"{label:>14} {hit:>8.4f} {hit / floor_hit:>8.1f}x")
print(f"\nCF vs popularity {cf_vs_popularity:.2f}x   vs lexical {cf_vs_lexical:.2f}x   vs floor {cf_vs_floor:.1f}x")

assert cf_hit > 1.3 * popularity_hit, "CF must beat popularity by a clear margin"
assert cf_hit > lexical_hit, "CF must beat the lexical path too (the first to beat BOTH)"
assert cf_vs_floor > 15, "CF must clear the random floor by a wide margin"
assert lexical_hit > popularity_hit, "honesty check: lexical already beats popularity, so CF is not first to"


## Exercise 7 - Challenge: the neighbour cap *degrades* the score here

The name "k-nearest-neighbour" suggests keeping only each book's top few neighbours and zeroing the rest -- a smaller, cleaner neighbourhood, offered by `ItemItemRetrievalPath` as an optional `n_neighbors` cap. On this data it is worth *measuring* rather than assuming, and the measurement says the cap **degrades** the score: with `n_neighbors=10` the hit@10 drops to about 0.142, below the uncapped 0.252 (and below even lexical). So the shipped default is uncapped.

In [ ]:
capped = bookrec.ItemItemRetrievalPath(n_neighbors=10).fit(rows, catalog=catalog_ids)
capped_hit = score(capped).hit_rate_at_k
uncapped_hit = cf_hit                                  # the uncapped default from Exercise 6
cap_degrades = capped_hit < uncapped_hit               # expect True: the cap HURTS here
cap_explanation = (
    "This log is sparse -- only a few positives per book -- so each book already has few non-zero "
    "neighbours; forcing a tight top-10 cap zeroes real co-occurrence edges that were carrying signal, "
    "and the summed-similarity vote loses the very neighbours that would have surfaced a reader's next "
    "book. With more edges thrown away than noise removed, the score falls, so the shipped default keeps "
    "the full (uncapped) neighbourhood."
)

print(f"uncapped item-item CF : {uncapped_hit:.4f}  (default)")
print(f"n_neighbors=10 cap    : {capped_hit:.4f}  <- lower than uncapped, and below lexical")
print(f"cap degrades the score: {cap_degrades}")

assert cap_degrades, "the n_neighbors=10 cap must score BELOW the uncapped default on this data"
assert capped_hit < lexical_hit, "the capped score even drops below the Unit-3 lexical path"


## Exercise 8 - Challenge: the coverage ceiling, and the low-rank bridge to matrix factorization

A collaborative path can only recommend a book that *someone* co-read: a book with no train positives is an all-zero column, gets an all-zero similarity row, and can never be a neighbour of anything. Using the library similarity `sim_lib` from Exercise 3, each book's total similarity is its row sum, and a book is **reachable** when that sum is positive. About $0.41$ of the catalog (818 of 2000 books) is unreachable. The honest interpretation has two halves: the ceiling mirrors the lexical path's cold-start strength (which is why Unit 6 blends them), and it bridges to Unit 5 through the **Gram matrix** $\mathbf{R}^{\top}\mathbf{R}$ -- whose rank is at most $\mathrm{rank}(\mathbf{R})$ -- which matrix factorization approximates with a *deliberately smaller* latent rank for the warm books, without reaching the cold ones.

In [ ]:
row_sums = sim_lib.sum(axis=1)                         # total similarity each book carries
reachable = int((row_sums > 0).sum())                  # books with some co-readership
unreachable = len(catalog_ids) - reachable             # books with no train positives
unreachable_share = unreachable / len(catalog_ids)     # expect ~0.41
coverage_explanation = (
    "The ~41% ceiling is the mirror image of the lexical path's strength: lexical can recommend a "
    "brand-new book from its words alone, with no readers at all, exactly where CF is blind, while CF "
    "wins on books the crowd has co-read. Because neither path covers the whole catalog by itself, the "
    "honest fix is to BLEND the collaborative and content candidates -- the cold-start work of Unit 6. "
    "The bridge to Unit 5 is a rank argument: the item-item similarity is the normalized Gram matrix "
    "R^T R of the reader-by-item incidence R, so its rank is at most rank(R); matrix factorization "
    "approximates that structure with a DELIBERATELY SMALLER latent rank (the low-rank view) that "
    "generalizes co-occurrence for the WARM books with interactions. It does NOT close this ceiling -- "
    "an id-only factorization still has no factors for the 818 cold books with zero train positives, "
    "which wait for Unit 6 blending and Unit 9 feature-based cold start."
)

print(f"reachable (have co-readership) : {reachable}")
print(f"unreachable (no train positives): {unreachable}  ({unreachable_share:.1%} of the catalog)")

assert reachable + unreachable == len(catalog_ids)
assert 0.35 < unreachable_share < 0.45, "about 41% of the catalog is unreachable by CF"
assert unreachable == 818
